# Ideal NMPC on the PrOMMiS mixer-settler

Rare earth elements are recovered from an acidic leach solution by
liquid-liquid extraction: the aqueous leachate is contacted with an
organic solvent, kerosene carrying the extractant DEHPA, which pulls the
rare earth ions across the phase boundary and leaves the impurities
behind. The model is the mixer-settler solvent extraction unit from
[PrOMMiS](https://github.com/prommis/prommis), built on IDAES, taken as
PrOMMiS wrote it: one extraction stage, with the feed composition of a
coal-refuse leachate carrying nine rare earths (Sc, Y, La, Ce, Pr, Nd,
Sm, Gd, Dy) at milligram-per-liter levels under percent-level Al, Ca,
and Fe impurities in sulfuric acid.

## The process

One stage is three vessels. The two phases meet only in the mixer, a
stirred tank both phases share; each effluent then passes through its
own settler, modeled as four well-mixed tanks in series, the
disengagement lag between the mixer and the outlets carried as a
residence-time distribution.

```
                    +--------------+       aqueous settler
 aqueous feed ----->|              |--aq-->[T1]-[T2]-[T3]-[T4]--> raffinate
 (leachate)         |    mixer     |
 organic feed ----->|  (2 phases)  |--org->[T1]-[T2]-[T3]-[T4]--> loaded solvent
 (kerosene+DEHPA)   |              |       organic settler
                    +--------------+
```

In the mixer the metals distribute between the phases through the DEHPA
complexation equilibria, metal ions trading places with protons, so
extraction pushes acid back into the aqueous phase. The aqueous effluent
(the raffinate) carries what was not extracted out through its settler
cascade; the organic effluent (the loaded solvent) carries the rare
earth complexes out through the other. In a multi-stage train the
organic stream runs backward through the stage sequence, making the
train counter-current; this example runs one stage.

## The dynamics

The states are inventories. With $n_j$ the molar holdup of species $j$
in the mixer, $V$ the mixer volume, $\varphi$ the aqueous volume
fraction, and $c_j$ the phase concentration, every mixer balance has the
same shape, indexed over the species of its phase:

$$\frac{dn_j}{dt} = F_{in}\, c_{j}^{feed} - F\, c_j
+ \sum_r \nu_{jr}\,\xi_r ,
\qquad n_j = V \varphi\, c_j ,$$

with $F$ the phase volumetric flow, $\nu_{jr}$ the stoichiometry, and
$\xi_r$ the extent of reaction $r$: the metal transfer complexations
and, in the aqueous phase, the bisulfate dissociation. The settlers are
written as transport along the vessel, discretized with backward
differences over four elements, and the backward scheme is exactly a
tank cascade: each element is a well-mixed tank of volume $A\,\Delta x$
fed by the one before it,

$$A\,\Delta x\,\frac{dc_{i}}{dt} = F\,c_{i-1} - F\,c_{i},$$

one balance per species per tank, transport only.

Not every inventory is a state. The transfer reactions are equilibria,
not rate laws, so the extents $\xi_r$ are algebraic and the organic
metal inventories follow the aqueous side instantaneously. The solvents
close by density ($c_{H2O} = \rho^{aq} - \sum_j c_j$ in the aqueous
phase, kerosene likewise in the organic), so their holdups carry no
independent memory of their own: in the settlers, where every tank runs
full at fixed volume, the solvent inventory is pinned outright; in the
mixer the water holdup moves only with the phase split $\varphi$, and
it is exactly the inventory that remembers the split. Bisulfate rides
its dissociation equilibrium. What remains is the memory the
optimization works with:

- the mixer's aqueous holdups, every species but bisulfate (the water
  member carrying the phase split),
- the mixer's free extractant holdup (DEHPA), the one organic inventory
  with memory of its own,
- the species holdups of each settler tank, four per settler, water,
  bisulfate, and kerosene excluded.

Everything else, concentrations, flows, the phase split, the extents,
is algebra the solver reconstructs at each instant.

## The declared model

[`models/prommis_sx.py`](models/prommis_sx.py) builds the stage and
declares the states above, both feed flows as the manipulated inputs
(with their operating limits as bounds), and additive zero-mean noise
terms in the state balances as the declared disturbances. The setpoint
is the steady operating point: the drto steady branch does not yet
reduce spatially distributed models, so the targets come from PrOMMiS's
own steady flowsheet, solved at the same dosage and read back through
the dynamic model's holdup equations.

In [ ]:
import pyomo.environ as pyo

import drto
from models.prommis_sx import build, steady_targets

m = build(N=8, h=0.25)
steady_targets(m)
drto.info(m)

## The infinite-horizon controller

An off-spec start: the mixer holds half again the rare earth
inventory of the operating point, an upstream upset that just cleared.
The terminal segment is appended, the model cold-started from the
off-spec state onto the targets, the optimization assembled, and the
horizon solved once: two hours of finite horizon and the infinite
tail carrying the rest.

In [ ]:
for j in ("Sc", "Y", "La", "Ce", "Pr", "Nd", "Sm", "Gd", "Dy"):
    m.ic_maq[j] = 1.5 * pyo.value(m.ic_maq[j])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
drto.cold_start_dynamic(m)
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)
res = pyo.SolverFactory("ipopt").solve(m, tee=True)
print(res.solver.termination_condition)

## The planned trajectories

The horizon drives the rare earth inventories from the upset back
toward the operating point, the two feed flows doing the work.

In [ ]:
axes = drto.plot_states(m)

In [ ]:
axes = drto.plot_controls(m)